# Exploring the supply chain data

DoD contract awards for bearings from USAspending.gov, as published by the
[supply chain ontology project](https://github.com/stschoberg/supply-chain-ontology). This notebook reads a
data release straight from GitHub. There is nothing to download or install.

- What the data can and can't tell you: [data/README.md](https://github.com/stschoberg/supply-chain-ontology/blob/main/data/README.md)
- Every column: [data/DICTIONARY.md](https://github.com/stschoberg/supply-chain-ontology/blob/main/data/DICTIONARY.md)

Run the cells in order (**Runtime → Run all** in Colab). Each query returns a pandas DataFrame.

In [ ]:
%pip install --quiet duckdb

## Connect

`data-latest` is the newest release. To reproduce a result, set `TAG` to a dated release such as
`data-2026-10-04`; those never change.

In [ ]:
import duckdb

TAG = "data-latest"
BASE = f"https://github.com/stschoberg/supply-chain-ontology/releases/download/{TAG}"

con = duckdb.connect()
con.sql(f"attach '{BASE}/catalog.duckdb' as sco (read_only)")
con.sql("use sco")


def q(sql):
    """Run a query against the release and return a pandas DataFrame."""
    return con.sql(sql).df()


q("select release_tag, git_sha, built_at from meta_build")

## What's in it

One table holds the data; the `meta_*` tables record where it came from. Each column carries its
description.

In [ ]:
q("""
    select column_name, data_type, comment as description
    from duckdb_columns()
    where database_name = 'sco' and table_name = 'stg_usaspending__awards'
""")

In [ ]:
q("""
    select award_key, recipient_name, psc, item_name, total_obligated_amount, base_action_date,
           offers_received, origin_country
    from stg_usaspending__awards
    limit 10
""")

## Who supplies DoD's bearings?

Identify suppliers by `recipient_uei`, not by name: names are spelled differently across awards.

In [ ]:
q("""
    select recipient_uei,
           any_value(recipient_name)         as supplier,
           count(*)                          as awards,
           sum(total_obligated_amount)::bigint as dollars
    from stg_usaspending__awards
    group by recipient_uei
    order by dollars desc
    limit 15
""")

## By product class

The Product and Service Code (`psc`) is the finest product identity that every award has:
3110 antifriction bearings, 3120 plain bearings, 3130 mounted bearings.

In [ ]:
q("""
    select psc,
           count(*)                                        as awards,
           count(distinct recipient_uei)                   as suppliers,
           sum(total_obligated_amount)::bigint             as dollars,
           round(100.0 * count(*) filter (where offers_received = 1)
                 / count(offers_received), 1)              as pct_single_offer
    from stg_usaspending__awards
    group by psc
    order by psc
""")

## Toward CQ-002: which products are single-sourced?

The ontology's [CQ-002](https://github.com/stschoberg/supply-chain-ontology/blob/main/competency-questions/CQ-002.md)
asks which products have exactly one supplier. Awards rarely say which part was bought, only a kind of
item (`item_name`), truncated and spelled inconsistently. So this answer is approximate: one kind of
item spelled two ways looks like two products.

In [ ]:
q("""
    select item_name,
           count(*)                      as awards,
           any_value(recipient_name)     as only_supplier
    from stg_usaspending__awards
    group by item_name
    having count(distinct recipient_uei) = 1
    order by awards desc
    limit 15
""")

The part-level answer needs the National Stock Number (NSN), which appears in under 1% of awards:

In [ ]:
q("""
    select nsn,
           any_value(description)            as description,
           count(*)                          as awards,
           count(distinct recipient_uei)     as suppliers
    from stg_usaspending__awards
    where nsn is not null
    group by nsn
    order by awards desc
    limit 15
""")

## Where do they come from?

Awards for products made outside the United States, by country of origin.

In [ ]:
q("""
    select origin_country,
           count(*)                          as awards,
           count(distinct recipient_uei)     as suppliers,
           sum(total_obligated_amount)::bigint as dollars
    from stg_usaspending__awards
    where origin_country <> 'UNITED STATES'
    group by origin_country
    order by awards desc
    limit 15
""")

## Your turn

Load the whole table into pandas, or write your own SQL with `q(...)`.

In [ ]:
awards = q("select * from stg_usaspending__awards")
awards.describe(include="all").T